# Coffee Quality Prediction - Exploratory Data Analysis

This notebook performs comprehensive EDA on the Coffee Quality Institute (CQI) dataset.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

In [ ]:
import sys
sys.path.insert(0, '../src')
from data_loader import DataLoader
from data_cleaner import DataCleaner

In [ ]:
# Load the datasets
loader = DataLoader()
arabica_df = loader.load_data('../data/arabica_data_cleaned.csv')
robusta_df = loader.load_data('../data/robusta_data_cleaned.csv')
print(f'Arabica dataset shape: {arabica_df.shape}')
print(f'Robusta dataset shape: {robusta_df.shape}')

In [ ]:
# Merge datasets
df = loader.merge_datasets(arabica_df, robusta_df)
print(f'Merged dataset shape: {df.shape}')

# Clean column names
cleaner = DataCleaner()
df = cleaner.clean_column_names(df)
print(f'\nColumn names after cleaning:')
print(df.columns.tolist())

In [ ]:
# Display dataset info
print('Dataset Information:')
print('=' * 50)
df.info()

In [ ]:
# Display first rows
print('First 5 rows of the dataset:')
df.head()

In [ ]:
# Display data types
print('Data Types:')
print('=' * 50)
print(df.dtypes)

## 2. Summary Statistics and Missing Value Analysis

In [ ]:
# Define numerical columns for analysis
sensory_cols = ['Aroma', 'Flavor', 'Aftertaste', 'Acidity', 'Body', 'Balance', 'Uniformity', 'Clean_Cup', 'Sweetness', 'Cupper_Points']
target_col = 'Total_Cup_Points'
numerical_cols = sensory_cols + [target_col, 'Moisture', 'altitude_mean_meters']

# Filter to columns that exist in the dataset
numerical_cols = [col for col in numerical_cols if col in df.columns]
print(f'Numerical columns for analysis: {numerical_cols}')

In [ ]:
# Summary statistics for numerical features
print('Summary Statistics for Numerical Features:')
print('=' * 80)
df[numerical_cols].describe().T

In [ ]:
# Missing value analysis
print('Missing Values Analysis:')
print('=' * 50)
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_counts, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print(missing_df)

In [ ]:
# Visualize missing values
fig, ax = plt.subplots(figsize=(12, 6))
if len(missing_df) > 0:
    missing_df['Missing %'].head(15).plot(kind='barh', ax=ax, color='coral')
    ax.set_xlabel('Missing Percentage (%)')
    ax.set_ylabel('Features')
    ax.set_title('Top 15 Features with Missing Values')
    plt.tight_layout()
else:
    ax.text(0.5, 0.5, 'No missing values found', ha='center', va='center', fontsize=14)
plt.show()

## 3. Target Variable Distribution Analysis

In [ ]:
# Target variable statistics
print('Target Variable (Total_Cup_Points) Statistics:')
print('=' * 50)
print(df['Total_Cup_Points'].describe())

In [ ]:
# Histogram of Total Cup Points
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['Total_Cup_Points'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(df['Total_Cup_Points'].mean(), color='red', linestyle='--', label=f'Mean: {df["Total_Cup_Points"].mean():.2f}')
axes[0].axvline(df['Total_Cup_Points'].median(), color='green', linestyle='--', label=f'Median: {df["Total_Cup_Points"].median():.2f}')
axes[0].set_xlabel('Total Cup Points')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Total Cup Points')
axes[0].legend()

# KDE plot
sns.kdeplot(data=df['Total_Cup_Points'].dropna(), ax=axes[1], fill=True, color='steelblue')
axes[1].set_xlabel('Total Cup Points')
axes[1].set_ylabel('Density')
axes[1].set_title('Density Plot of Total Cup Points')

plt.tight_layout()
plt.show()

In [ ]:
# Create quality categories
def categorize_quality(score):
    if pd.isna(score):
        return 'Unknown'
    elif score >= 90:
        return 'Outstanding (90+)'
    elif score >= 85:
        return 'Excellent (85-90)'
    elif score >= 80:
        return 'Very Good (80-85)'
    elif score >= 75:
        return 'Good (75-80)'
    else:
        return 'Below Average (<75)'

df['Quality_Category'] = df['Total_Cup_Points'].apply(categorize_quality)
print('Quality Category Distribution:')
print(df['Quality_Category'].value_counts())

In [ ]:
# Box plot by quality categories
fig, ax = plt.subplots(figsize=(10, 6))
category_order = ['Below Average (<75)', 'Good (75-80)', 'Very Good (80-85)', 'Excellent (85-90)', 'Outstanding (90+)']
existing_categories = [cat for cat in category_order if cat in df['Quality_Category'].values]
sns.boxplot(data=df[df['Quality_Category'] != 'Unknown'], x='Quality_Category', y='Total_Cup_Points', order=existing_categories, ax=ax, palette='viridis')
ax.set_xlabel('Quality Category')
ax.set_ylabel('Total Cup Points')
ax.set_title('Total Cup Points Distribution by Quality Category')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
# Correlation matrix for sensory attributes
sensory_cols_existing = [col for col in sensory_cols if col in df.columns]
correlation_cols = sensory_cols_existing + ['Total_Cup_Points']
corr_matrix = df[correlation_cols].corr()

print('Correlation Matrix for Sensory Attributes:')
print('=' * 50)
print(corr_matrix.round(3))

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r', center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Heatmap: Sensory Attributes and Total Cup Points')
plt.tight_layout()
plt.show()

In [ ]:
# Correlations with target variable
target_corr = df[correlation_cols].corr()['Total_Cup_Points'].drop('Total_Cup_Points').sort_values(ascending=False)
print('Correlations with Total Cup Points:')
print('=' * 50)
print(target_corr)

In [ ]:
# Bar plot of correlations with target
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['green' if x > 0 else 'red' for x in target_corr.values]
target_corr.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('Correlation Coefficient')
ax.set_ylabel('Features')
ax.set_title('Feature Correlations with Total Cup Points')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots of top correlated features vs target
top_features = target_corr.head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    ax = axes[i]
    ax.scatter(df[feature], df['Total_Cup_Points'], alpha=0.5, s=20)
    # Add trend line
    z = np.polyfit(df[feature].dropna(), df.loc[df[feature].notna(), 'Total_Cup_Points'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df[feature].min(), df[feature].max(), 100)
    ax.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Trend (r={target_corr[feature]:.3f})')
    ax.set_xlabel(feature)
    ax.set_ylabel('Total Cup Points')
    ax.set_title(f'{feature} vs Total Cup Points')
    ax.legend()

plt.tight_layout()
plt.show()

## 5. Categorical Feature Analysis

In [ ]:
# Identify categorical columns
categorical_cols = ['Country_of_Origin', 'Processing_Method', 'Variety', 'Species', 'Color']
categorical_cols = [col for col in categorical_cols if col in df.columns]
print(f'Categorical columns for analysis: {categorical_cols}')

In [ ]:
# Quality distribution by Country of Origin
if 'Country_of_Origin' in df.columns:
    country_quality = df.groupby('Country_of_Origin')['Total_Cup_Points'].agg(['mean', 'count']).sort_values('mean', ascending=False)
    country_quality = country_quality[country_quality['count'] >= 10]  # Filter countries with at least 10 samples
    
    print('Average Quality by Country (min 10 samples):')
    print('=' * 50)
    print(country_quality.head(15))

In [ ]:
# Bar chart - Top countries by average quality
if 'Country_of_Origin' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Top 10 countries by average quality
    top_countries = country_quality.head(10)
    axes[0].barh(top_countries.index, top_countries['mean'], color='steelblue')
    axes[0].set_xlabel('Average Total Cup Points')
    axes[0].set_ylabel('Country')
    axes[0].set_title('Top 10 Countries by Average Coffee Quality')
    
    # Sample count by country
    country_counts = df['Country_of_Origin'].value_counts().head(10)
    axes[1].barh(country_counts.index, country_counts.values, color='coral')
    axes[1].set_xlabel('Number of Samples')
    axes[1].set_ylabel('Country')
    axes[1].set_title('Top 10 Countries by Sample Count')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Box plot - Quality by top countries
if 'Country_of_Origin' in df.columns:
    top_10_countries = df['Country_of_Origin'].value_counts().head(10).index.tolist()
    df_top_countries = df[df['Country_of_Origin'].isin(top_10_countries)]
    
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.boxplot(data=df_top_countries, x='Country_of_Origin', y='Total_Cup_Points', ax=ax, palette='Set2')
    ax.set_xlabel('Country of Origin')
    ax.set_ylabel('Total Cup Points')
    ax.set_title('Coffee Quality Distribution by Country (Top 10 by Sample Count)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Quality distribution by Processing Method
if 'Processing_Method' in df.columns:
    processing_quality = df.groupby('Processing_Method')['Total_Cup_Points'].agg(['mean', 'count', 'std']).sort_values('mean', ascending=False)
    print('Average Quality by Processing Method:')
    print('=' * 50)
    print(processing_quality)

In [ ]:
# Box plot - Quality by Processing Method
if 'Processing_Method' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    df_processing = df[df['Processing_Method'].notna()]
    sns.boxplot(data=df_processing, x='Processing_Method', y='Total_Cup_Points', ax=ax, palette='Set3')
    ax.set_xlabel('Processing Method')
    ax.set_ylabel('Total Cup Points')
    ax.set_title('Coffee Quality Distribution by Processing Method')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Quality distribution by Variety
if 'Variety' in df.columns:
    variety_quality = df.groupby('Variety')['Total_Cup_Points'].agg(['mean', 'count']).sort_values('mean', ascending=False)
    variety_quality = variety_quality[variety_quality['count'] >= 5]  # Filter varieties with at least 5 samples
    
    print('Average Quality by Variety (min 5 samples):')
    print('=' * 50)
    print(variety_quality.head(15))

In [ ]:
# Box plot - Quality by top varieties
if 'Variety' in df.columns:
    top_varieties = df['Variety'].value_counts().head(8).index.tolist()
    df_top_varieties = df[df['Variety'].isin(top_varieties)]
    
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=df_top_varieties, x='Variety', y='Total_Cup_Points', ax=ax, palette='husl')
    ax.set_xlabel('Variety')
    ax.set_ylabel('Total Cup Points')
    ax.set_title('Coffee Quality Distribution by Variety (Top 8 by Sample Count)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Feature Importance Analysis

In [ ]:
# Prepare data for feature importance analysis
# Use only sensory attributes for the preliminary model
feature_cols = [col for col in sensory_cols_existing if col in df.columns]
print(f'Features for importance analysis: {feature_cols}')

# Create a clean dataset for modeling
df_model = df[feature_cols + ['Total_Cup_Points']].dropna()
print(f'Samples for modeling: {len(df_model)}')

In [ ]:
# Train a Random Forest model for feature importance
X = df_model[feature_cols]
y = df_model['Total_Cup_Points']

rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X, y)

# Get feature importances
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Feature Importance (Random Forest):')
print('=' * 50)
print(feature_importance)
print(f'\nSum of importances: {feature_importance["Importance"].sum():.4f}')

In [ ]:
# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.viridis(np.linspace(0, 1, len(feature_importance)))
ax.barh(feature_importance['Feature'], feature_importance['Importance'], color=colors)
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
ax.set_title('Feature Importance for Coffee Quality Prediction (Random Forest)')
ax.invert_yaxis()  # Highest importance at top
plt.tight_layout()
plt.show()

In [ ]:
# Compare correlation vs feature importance
comparison_df = pd.DataFrame({
    'Feature': feature_cols,
    'Correlation': [target_corr.get(f, 0) for f in feature_cols],
    'RF_Importance': [feature_importance[feature_importance['Feature'] == f]['Importance'].values[0] for f in feature_cols]
})
comparison_df = comparison_df.sort_values('RF_Importance', ascending=False)

print('Comparison: Correlation vs Random Forest Importance:')
print('=' * 60)
print(comparison_df)

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison_df))
width = 0.35

bars1 = ax.bar(x - width/2, comparison_df['Correlation'], width, label='Correlation', color='steelblue')
bars2 = ax.bar(x + width/2, comparison_df['RF_Importance'], width, label='RF Importance', color='coral')

ax.set_xlabel('Features')
ax.set_ylabel('Score')
ax.set_title('Feature Correlation vs Random Forest Importance')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Feature'], rotation=45, ha='right')
ax.legend()
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

## 7. Summary and Key Findings

In [ ]:
# Summary statistics
print('=' * 60)
print('EDA SUMMARY')
print('=' * 60)
print(f'\nDataset Size: {len(df)} samples')
print(f'Number of Features: {len(df.columns)}')
print(f'\nTarget Variable (Total Cup Points):')
print(f'  - Mean: {df["Total_Cup_Points"].mean():.2f}')
print(f'  - Std: {df["Total_Cup_Points"].std():.2f}')
print(f'  - Range: {df["Total_Cup_Points"].min():.2f} - {df["Total_Cup_Points"].max():.2f}')
print(f'\nTop 3 Most Important Features (RF):')
for i, row in feature_importance.head(3).iterrows():
    print(f'  - {row["Feature"]}: {row["Importance"]:.4f}')
print(f'\nTop 3 Most Correlated Features:')
for feat, corr in target_corr.head(3).items():
    print(f'  - {feat}: {corr:.4f}')